In [ ]:
import pandas as pd
import numpy as np
import glob
import os
import datetime
import re

pd.set_option('display.max_columns', 75)
pd.set_option('display.float_format', '{:.2f}'.format)

# 1. Считать все файлы csv выгрузки из СБИС в единый DF
files = glob.glob('Входящие/*.csv', recursive=True)  
skiprow = True # флаг для пропуска заголовка в последйющих данных

for file in files:
    if skiprow:
        first_df = pd.read_csv(file, sep=';', encoding='1251')
        df_SBIS = first_df.copy()
        skiprow = False
    else:
        next_df = pd.read_csv(file, sep=';', encoding='1251')
        df_SBIS = pd.concat([df_SBIS, next_df], axis=0)
        
# 2. Привести названия столбцов в соответствие с заданием
df_SBIS.columns = ["Дата","Номер","Сумма","Статус","Примечание","Комментарий","Контрагент","ИНН_КПП","Организация","ИНН_КПП","Тип документа","Имя файла",
"Дата","Номер_1","Сумма_1","Сумма_НДС","Ответственный","Подразделение","Код","Дата","Время","Тип_пакета","Идентификатор_пакета","Запущено_в_обработку",
"Получено_контрагентом","Завершено","Увеличение_суммы","НДC","Уменьшение_суммы","НДС"]


# 3. Проверить наличие итоговой папки для записи результатов, если нет, создать ее
result_dir = os.path.join('Результат', datetime.date.today().strftime('%Y-%m-%d'))
os.makedirs(result_dir, exist_ok=True)

# 4. Выгрузка по аптекам
files = glob.glob('Аптеки/csv/correct/*.csv', recursive=True)
for file in files:
    # Достать название файла
    file_name = (re.search(r"[^\\]+(?=\.csv)", str(file)).group(0))
    
    # 4. Считать файл
    df = pd.read_csv(file, sep=';',encoding='1251')
    
    # 5. Добавить необходимые поля
    df[["Номер счет-фактуры", "Сумма счет-фактуры", "Дата счет-фактуры", "Сравнение дат"]] = np.nan
    
    # 6. Проверить поставщиков
    df['Поставщик'] = np.where(df['Поставщик']=='ЕАПТЕКА', df['Номер накладной']+'/15', df['Номер накладной'])
    # 6.1 Пробежаться по все накладным
    for nak in df['Номер накладной']:
        if len(df_SBIS[(df_SBIS['Номер']==nak) & (df_SBIS['Тип документа'].isin(["СчФктр", "УпдДоп", "УпдСчфДоп", "ЭДОНакл"]))]) == 0: # если ничего не найдено - пропускаем
            pass
        else:
            need_data = df_SBIS[(df_SBIS['Номер']==nak) & (df_SBIS['Тип документа'].isin(["СчФктр", "УпдДоп", "УпдСчфДоп", "ЭДОНакл"]))].iloc[0,[0,1,2]]
            need_data['Дата'] = pd.to_datetime(need_data['Дата'], dayfirst=True).strftime('%d.%m.%Y') # преобразовать формат даты
        
        # преобразование типов для исключения ошибок в будущемv
        df.loc[:,'Номер счет-фактуры'] = df['Номер счет-фактуры'].astype('string') 
        df.loc[:,'Сумма счет-фактуры'] = df['Сумма счет-фактуры'].astype('string')
        df.loc[:,'Дата счет-фактуры'] = df['Дата счет-фактуры'].astype('string')
        
        # запись данных
        df.loc[df['Номер накладной']==nak, 'Номер счет-фактуры'] = need_data['Номер']
        df.loc[df['Номер накладной']==nak, 'Сумма счет-фактуры'] = need_data['Сумма']
        df.loc[df['Номер накладной']==nak, 'Дата счет-фактуры'] = need_data['Дата']
        # сравнение дат
        df.loc[df['Номер накладной']==nak, 'Сравнение дат'] = np.where(df.loc[df['Номер накладной']==nak, 'Дата накладной'] != df.loc[df['Номер накладной']==nak, 'Дата счет-фактуры'], 'Не совпадает!', '')
        
    # 7. Привести расположение полей в соответствие
    df = df.loc[:,['№ п/п', 'Штрих-код партии', 'Наименование товара', 'Поставщик',
    'Дата приходного документа', 'Номер приходного документа',
    'Дата накладной', 'Номер накладной', 'Номер счет-фактуры',
    'Сумма счет-фактуры', 'Кол-во',
    'Сумма в закупочных ценах без НДС', 'Ставка НДС поставщика',
    'Сумма НДС', 'Сумма в закупочных ценах с НДС', 'Дата счет-фактуры', 'Сравнение дат']]

    # 8. Экспорт данных в xlsx
    df.to_excel(
        f"{result_dir}\\{file_name} - результат.xlsx",
        index = False
    )